# TrendPulse — Part 2: NumPy & Pandas Data Wrangling

Goal: turn three countries of raw daily trending lists into an analysis-ready table, compute a transparent **engagement score** with
NumPy (no loops), and answer concrete questions about what trends and for how long.

Everything is read from this folder's `data/` subfolder; no network access is used.

**Sections**
1. Loading and inspection (per country and combined)
2. Data-quality issues, missing values and duplicates
3. NumPy arrays — shape, dtype, indexing and slicing
4. Engagement score without loops
5. Reshape and broadcasting (days × countries)
6. Joining category names
7. Filtering
8. Aggregation
9. Time to trend and trending lifespan
10. Combining data (concat + merge)

*Note on order:* the brief lists the NumPy tasks first. I load and clean with Pandas first so that the NumPy arrays are built from clean
values (no duplicates, no removed videos, no NaN), then attach the NumPy results back to the DataFrame.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

DATA_DIR = "data"
COUNTRIES = ["US", "GB", "IN"]
print("numpy", np.__version__, "| pandas", pd.__version__)

numpy 1.24.4 | pandas 2.0.3


## 1. Loading and inspection

Each country file is read with `pd.read_csv()`, given a `country` column and inspected with `shape`, `head()`, `info()` and
`describe()`.

In [2]:
frames = {}
for c in COUNTRIES:
    df_c = pd.read_csv(f"{DATA_DIR}/{c}videos.csv")
    df_c["country"] = c
    frames[c] = df_c
    print(f"{c}: shape = {df_c.shape}")

US: shape = (16080, 17)
GB: shape = (15075, 17)
IN: shape = (15075, 17)


In [3]:
for c in COUNTRIES:
    print(f"===== {c}: head()")
    display(frames[c][["video_id", "trending_date", "title", "category_id", "views", "likes", "dislikes", "comment_count"]].head())

===== US: head()


,video_id,trending_date,title,category_id,views,likes,dislikes,comment_count
0,g_N-w8LeAi8,17.14.11,Honest Football Prank LEAKED,22,48555,"2,189.000",180,136.000
1,Ugyql2NlPAd,17.14.11,Surprising Cricket Review,25,198385,"1,159.000",492,"1,446.000"
2,8fE19jkxJ9n,17.14.11,Mira Solis - Football (Live) OFFICIAL,10,231656,"8,742.000",896,432.000
3,WOY0nn7HCsO,17.14.11,Zed Rivera - Space (Lyric Video)?!,10,8083,176.000,8,19.000
4,2Lc_Cs5VVr_,17.14.11,Surprising Election Recipe,27,373161,"11,305.000",167,891.000


===== GB: head()


,video_id,trending_date,title,category_id,views,likes,dislikes,comment_count
0,u_nKn025TZS,17.14.11,Wild Winter Prank,28,84344,"2,155.000",163,475.000
1,FoQAEGxMMQD,17.14.11,Wild Cricket Tutorial FINALE,26,230434,"5,412.000",330,"1,704.000"
2,wHqDBuStXq3,17.14.11,Epic Space Challenge,23,338852,"16,813.000",1333,"1,385.000"
3,ZId0IGzSaaZ,17.14.11,Quiet Paris Prank LIVE?!,24,160896,"4,221.000",483,"1,342.000"
4,1-xA4eRrYSH,17.14.11,Nova Kane - Chess (Official Video)!,10,59441,"1,475.000",77,60.000


===== IN: head()


,video_id,trending_date,title,category_id,views,likes,dislikes,comment_count
0,UcYiQY22yfG,17.14.11,Sunny Park - Cats (Lyric Video)!,10,217915,"14,918.000",1363,662.000
1,EutcTEIciae,17.14.11,Perfect Pasta Unboxing?!,1,115014,"2,739.000",287,209.000
2,rvuZ5CsBpg8,17.14.11,Wild Budget Challenge!!,24,51028,"1,786.000",322,365.000
3,2yJGS3uxgMP,17.14.11,Quiet Chess Mystery,23,143370,"5,224.000",305,571.000
4,W5pdmLZu7GK,17.14.11,Quiet Festival Prank,24,390220,"15,079.000",947,"1,942.000"


In [4]:
for c in COUNTRIES:
    print(f"===== {c}: info()")
    frames[c].info()
    print()

===== US: info()
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16080 entries, 0 to 16079
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                16080 non-null  object 
 1   trending_date           16080 non-null  object 
 2   title                   16080 non-null  object 
 3   channel_title           16080 non-null  object 
 4   category_id             16080 non-null  int64  
 5   publish_time            16080 non-null  object 
 6   tags                    16080 non-null  object 
 7   views                   16080 non-null  int64  
 8   likes                   16076 non-null  float64
 9   dislikes                16080 non-null  int64  
 10  comment_count           16068 non-null  float64
 11  thumbnail_link          16080 non-null  object 
 12  comments_disabled       16080 non-null  bool   
 13  ratings_disabled        16080 non-null  bool   
 14  video_error_or_remove

In [5]:
for c in COUNTRIES:
    print(f"===== {c}: describe()")
    display(frames[c][["views", "likes", "dislikes", "comment_count"]].describe())

===== US: describe()


,views,likes,dislikes,comment_count
count,"16,080.000","16,076.000","16,080.000","16,068.000"
mean,"1,012,289.202","39,504.638","2,115.772","3,118.738"
std,"3,746,760.857","171,941.394","7,973.928","10,060.176"
min,"1,621.000",0.000,0.000,0.000
25%,"117,307.750","3,353.000",194.000,375.000
50%,"289,072.500","9,229.000",540.000,999.000
75%,"750,719.000","26,267.000","1,506.000","2,613.250"
max,"129,229,245.000","6,795,138.000","247,546.000","318,346.000"


===== GB: describe()


,views,likes,dislikes,comment_count
count,"15,075.000","15,071.000","15,075.000","15,063.000"
mean,"1,580,015.493","65,097.000","2,938.435","4,042.069"
std,"9,105,598.442","476,940.047","15,407.807","19,902.746"
min,"1,066.000",0.000,0.000,0.000
25%,"121,176.500","3,658.000",194.000,356.000
50%,"316,900.000","10,537.000",569.000,977.000
75%,"867,429.500","31,800.000","1,688.500","2,632.500"
max,"360,760,164.000","27,537,822.000","618,175.000","778,390.000"


===== IN: describe()


,views,likes,dislikes,comment_count
count,"15,075.000","15,071.000","15,075.000","15,063.000"
mean,"954,592.447","33,218.333","1,937.758","3,210.599"
std,"3,035,646.781","104,210.317","5,791.483","8,251.638"
min,"3,542.000",0.000,0.000,0.000
25%,"152,165.500","4,107.000",257.000,496.000
50%,"355,802.000","10,915.000",681.000,"1,243.000"
75%,"863,998.000","29,262.000","1,815.500","3,101.000"
max,"123,208,090.000","3,634,665.000","287,063.000","346,103.000"


### Combined data
`pd.concat()` stacks the three country frames vertically (same columns). `ignore_index=True` gives the combined frame a fresh
0…n-1 index instead of three overlapping ones.

In [6]:
raw = pd.concat([frames[c] for c in COUNTRIES], ignore_index=True)
print("combined shape:", raw.shape)
print("sum of the parts:", sum(len(frames[c]) for c in COUNTRIES))
display(raw.head())
raw.info()
display(raw.describe())

combined shape: (46230, 17)
sum of the parts: 46230


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,g_N-w8LeAi8,17.14.11,Honest Football Prank LEAKED,QuietSpeech10,22,2017-11-14T07:04:28.000Z,Monsoon|Robots,48555,"2,189.000",180,136.000,https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
1,Ugyql2NlPAd,17.14.11,Surprising Cricket Review,IncredibleChallenge41,25,2017-11-12T22:35:48.000Z,Winter,198385,"1,159.000",492,"1,446.000",https://i.ytimg.com/vi/Ugyql2NlPAd/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
2,8fE19jkxJ9n,17.14.11,Mira Solis - Football (Live) OFFICIAL,WildTutorial42,10,2017-11-13T08:40:22.000Z,Cricket,231656,"8,742.000",896,432.000,https://i.ytimg.com/vi/8fE19jkxJ9n/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
3,WOY0nn7HCsO,17.14.11,Zed Rivera - Space (Lyric Video)?!,ShockingInterview30,10,2017-10-27T22:21:47.000Z,Festival|Guitar,8083,176.000,8,19.000,https://i.ytimg.com/vi/WOY0nn7HCsO/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
4,2Lc_Cs5VVr_,17.14.11,Surprising Election Recipe,UltimateVlog96,27,2017-11-09T13:20:20.000Z,Festival,373161,"11,305.000",167,891.000,https://i.ytimg.com/vi/2Lc_Cs5VVr_/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46230 entries, 0 to 46229
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                46230 non-null  object 
 1   trending_date           46230 non-null  object 
 2   title                   46230 non-null  object 
 3   channel_title           46230 non-null  object 
 4   category_id             46230 non-null  int64  
 5   publish_time            46230 non-null  object 
 6   tags                    46230 non-null  object 
 7   views                   46230 non-null  int64  
 8   likes                   46218 non-null  float64
 9   dislikes                46230 non-null  int64  
 10  comment_count           46194 non-null  float64
 11  thumbnail_link          46230 non-null  object 
 12  comments_disabled       46230 non-null  bool   
 13  ratings_disabled        46230 non-null  bool   
 14  video_error_or_removed  46230 non-null

,category_id,views,likes,dislikes,comment_count
count,"46,230.000","46,230.000","46,218.000","46,230.000","46,194.000"
mean,18.645,"1,178,603.181","45,800.055","2,325.984","3,449.773"
std,7.647,"5,916,188.240","296,955.393","10,518.963","13,665.153"
min,1.000,"1,066.000",0.000,0.000,0.000
25%,10.000,"129,463.250","3,656.000",213.000,404.000
50%,22.000,"318,277.500","10,201.000",594.000,"1,074.000"
75%,24.000,"824,050.500","28,987.750","1,666.000","2,779.000"
max,28.000,"360,760,164.000","27,537,822.000","618,175.000","778,390.000"


### Fixing data types
* `trending_date` is stored as text in **`yy.dd.mm`** order (`17.14.11` = 14 Nov 2017) → `pd.to_datetime(format="%y.%d.%m")`.
* `publish_time` is an ISO-8601 UTC string → `pd.to_datetime(utc=True)`, then made timezone-naive so it can be compared with the
  trending date.
* `likes` and `comment_count` were read as `float64` only because they contain blanks; they become integers after missing values are
  handled (section 2).
* `category_id` stays an integer key; `country` becomes a categorical.

In [7]:
df = raw.copy()
df["trending_date"] = pd.to_datetime(df["trending_date"], format="%y.%d.%m", errors="coerce")
df["publish_time"] = pd.to_datetime(df["publish_time"], errors="coerce", utc=True).dt.tz_convert(None)
df["country"] = df["country"].astype("category")

print("unparseable trending_date:", df["trending_date"].isna().sum())
print("unparseable publish_time :", df["publish_time"].isna().sum())
print("trending_date range:", df["trending_date"].min().date(), "->", df["trending_date"].max().date())
print(df.groupby("country", observed=True)["trending_date"].agg(["min", "max", "nunique"]))

unparseable trending_date: 0
unparseable publish_time : 0
trending_date range: 2017-11-14 -> 2018-02-01
               min        max  nunique
country                               
GB      2017-11-14 2018-01-27       75
IN      2017-11-14 2018-01-27       75
US      2017-11-14 2018-02-01       80


## 2. Data-quality issues, missing values and duplicates

### 2.1 Missing values per column

In [8]:
missing = df.isna().sum()
missing = missing[missing > 0].to_frame("missing")
missing["percent"] = (missing["missing"] / len(df) * 100).round(3)
missing

,missing,percent
likes,12,0.026
comment_count,36,0.078
description,1624,3.513


| Column | Strategy | Justification |
|---|---|---|
| `description` | `fillna("")` | Free text that no calculation uses; an empty string honestly means "no description" and keeps the row. |
| `likes` | fill from the **same video's** neighbouring trending day in the same country (`groupby` + forward/backward fill, passed to `fillna`), then `dropna` for anything left | Counts move smoothly from day to day for one video, so the neighbouring day is a far better estimate than a global mean or 0 (0 would wrongly make the video look unliked). |
| `comment_count` | same as `likes` | same reason; 0 would be confused with `comments_disabled`. |

The other "missing-like" values are **not** missing data: `tags == "[none]"` means the uploader added no tags, and zero likes/dislikes
for `ratings_disabled` videos (and zero comments for `comments_disabled` videos) are genuine zeros produced by the platform.

### 2.2 Duplicates
A **duplicate** is the same `video_id` on the same `trending_date` in the same `country`: a trending list contains a video only once per
day, so a second row with that key is the same observation recorded twice. I check both exact duplicates and duplicates on that key.

In [9]:
KEY = ["country", "video_id", "trending_date"]
print("exact duplicate rows      :", df.duplicated().sum())
print("duplicates on the key     :", df.duplicated(subset=KEY).sum())
before = len(df)
df = df.drop_duplicates(subset=KEY, keep="first")
print(f"removed {before - len(df)} duplicate rows -> {len(df):,} rows")
print(df.groupby("country", observed=True).size())

exact duplicate rows      : 230
duplicates on the key     : 230


removed 230 duplicate rows -> 46,000 rows
country
GB    15000
IN    15000
US    16000
dtype: int64


### 2.3 Removed / errored videos

In [10]:
removed = df["video_error_or_removed"]
print("records flagged video_error_or_removed:", removed.sum())
df = df[~removed].copy()
print("rows after dropping them:", len(df))

records flagged video_error_or_removed: 92


rows after dropping them: 45908


### 2.4 Filling the missing counts

In [11]:
VIDEO = ["country", "video_id"]
df = df.sort_values(KEY).reset_index(drop=True)

df["description"] = df["description"].fillna("")
for col in ["likes", "comment_count"]:
    neighbour = df.groupby(VIDEO, observed=True)[col].transform(lambda s: s.ffill().bfill())
    filled = df[col].isna() & neighbour.notna()
    df[col] = df[col].fillna(neighbour)
    print(f"{col}: filled {filled.sum()} values from the same video's neighbouring day")

before = len(df)
df = df.dropna(subset=["likes", "comment_count"])
print(f"dropna removed {before - len(df)} rows that had no neighbouring day to borrow from")

df["likes"] = df["likes"].astype("int64")
df["comment_count"] = df["comment_count"].astype("int64")
print("remaining missing values:", int(df.isna().sum().sum()))
print("clean shape:", df.shape)

likes: filled 10 values from the same video's neighbouring day


comment_count: filled 34 values from the same video's neighbouring day
dropna removed 4 rows that had no neighbouring day to borrow from
remaining missing values: 0
clean shape: (45904, 17)


### 2.5 Other checks

In [12]:
checks = {
    "negative counts": int((df[["views", "likes", "dislikes", "comment_count"]] < 0).any(axis=1).sum()),
    "published AFTER trending day": int((df["publish_time"].dt.normalize() > df["trending_date"]).sum()),
    "ratings_disabled with likes/dislikes > 0": int((df["ratings_disabled"] & ((df["likes"] > 0) | (df["dislikes"] > 0))).sum()),
    "comments_disabled with comments > 0": int((df["comments_disabled"] & (df["comment_count"] > 0)).sum()),
    "tags == '[none]'": int((df["tags"] == "[none]").sum()),
    "videos whose title differs between rows": int((df.groupby("video_id")["title"].nunique() > 1).sum()),
}
for k, v in checks.items():
    print(f"{k:<44} {v:>6}")

negative counts                                   0
published AFTER trending day                      0
ratings_disabled with likes/dislikes > 0          0
comments_disabled with comments > 0               0
tags == '[none]'                               3763
videos whose title differs between rows           0


### 2.6 Data-quality issues found and how each was handled

| # | Issue | Size | Handling |
|---|---|---|---|
| 1 | Exact duplicate rows (same country, video, day) | 230 rows | `drop_duplicates(subset=KEY)` — kept the first |
| 2 | Removed / errored videos | 92 rows | dropped: their statistics are not valid |
| 3 | Missing `description` | ≈3.5% of rows (see table 2.1) | `fillna("")` |
| 4 | Missing `likes` | 12 rows | filled from the same video's neighbouring day; any leftovers dropped |
| 5 | Missing `comment_count` | 36 rows | same as likes |
| 6 | `likes`/`comment_count` stored as floats (`2155.0`) | whole columns | cast to `int64` after filling |
| 7 | `trending_date` in an unusual `yy.dd.mm` text format | whole column | parsed with an explicit format; 0 failures |
| 8 | `publish_time` as UTC text | whole column | parsed to datetime, timezone removed |
| 9 | `tags == "[none]"` placeholder | see 2.5 | kept — it means "no tags", not missing data |
| 10 | Zero likes/comments when ratings/comments are disabled | see flags | kept as genuine zeros; ratios handle the zero denominators |
| 11 | Countries cover different windows (US 80 days, GB/IN 75 days) | dates | kept; per-country comparisons use averages, and the days × countries array uses only the common days |

The exact sizes for this run are printed in the cells above.

## 3. NumPy arrays

The four count columns are pulled out of the clean DataFrame as NumPy arrays.

In [13]:
views = df["views"].to_numpy()
likes = df["likes"].to_numpy()
dislikes = df["dislikes"].to_numpy()
comments = df["comment_count"].to_numpy()

for name, arr in [("views", views), ("likes", likes), ("dislikes", dislikes), ("comments", comments)]:
    print(f"{name:<9} shape={arr.shape}  dtype={arr.dtype}  min={arr.min():>9,}  max={arr.max():>12,}")

views     shape=(45904,)  dtype=int64  min=    1,066  max= 360,760,164
likes     shape=(45904,)  dtype=int64  min=        0  max=  27,537,822
dislikes  shape=(45904,)  dtype=int64  min=        0  max=     618,175
comments  shape=(45904,)  dtype=int64  min=        0  max=     778,390


In [14]:
top10_views = np.sort(views)[-10:][::-1]          # sort ascending, take the last 10, reverse
print("10 largest view counts:", top10_views)

every_100th = views[::100]
print(f"every 100th record: shape={every_100th.shape}, first 10 = {every_100th[:10]}")

stacked = np.column_stack([views, likes, dislikes, comments])   # (n, 4) 2-D array
print("stacked shape:", stacked.shape, "dtype:", stacked.dtype)
print("sub-array rows 1000-1004, columns likes..comments:")
print(stacked[1000:1005, 1:4])

10 largest view counts: [360760164 343297002 286714357 282812154 255362869 236275528 230704137
 210747294 200444300 181785943]
every 100th record: shape=(460,), first 10 = [  96088   73156  267948 2645280 1135050  419223  484818  289905  163926
  312285]
stacked shape: (45904, 4) dtype: int64
sub-array rows 1000-1004, columns likes..comments:
[[12226   155   453]
 [ 4104   180   829]
 [ 5084   223  1028]
 [ 6563   288  1327]
 [ 7828   343  1582]]


## 4. Engagement score without loops

All steps below are whole-array NumPy operations — there is no Python `for`/`while` loop.

**(a) Ratios with safe denominators.** `np.divide(a, b, out=..., where=b > 0)` only divides where the denominator is positive and
leaves the prepared default everywhere else:
* like ratio = likes / (likes + dislikes); with no votes (e.g. ratings disabled) the default is **NaN** ("unknown"), and the score uses a
  neutral 0.5 for it, so a video is neither rewarded nor punished for having ratings off;
* comment rate = comments / views; default 0 (no views ⇒ no comment rate).

In [15]:
votes = likes + dislikes
like_ratio = np.divide(likes, votes, out=np.full(views.shape, np.nan), where=votes > 0)
comment_rate = np.divide(comments, views, out=np.zeros(views.shape), where=views > 0)

print("like ratio  : NaN (no votes) =", np.isnan(like_ratio).sum(), "| median =", round(np.nanmedian(like_ratio), 4))
print("comment rate: median =", round(np.median(comment_rate), 5), "| max =", round(comment_rate.max(), 5))

like ratio  : NaN (no votes) = 539 | median = 0.9489
comment rate: median = 0.0033 | max = 0.02463


**(b) Min-max normalisation.** `x_norm = (x − min) / (max − min)` maps a column to [0, 1].

On the raw counts this is badly squashed: one record with ~90 M views sets the maximum, so the median record ends up near 0.
I therefore also normalise the **log** of each count (`np.log1p`, which is defined at 0). The log version spreads the records over [0, 1]
and is the one used in the score.

In [16]:
def minmax(a):
    a = a.astype("float64")
    span = a.max() - a.min()
    return (a - a.min()) / span if span > 0 else np.zeros_like(a)

views_n_raw, likes_n_raw, comments_n_raw = minmax(views), minmax(likes), minmax(comments)
views_n, likes_n, comments_n = minmax(np.log1p(views)), minmax(np.log1p(likes)), minmax(np.log1p(comments))

print(f"{'':<10}{'raw median':>12}{'log median':>12}")
for name, r, l in [("views", views_n_raw, views_n), ("likes", likes_n_raw, likes_n), ("comments", comments_n_raw, comments_n)]:
    print(f"{name:<10}{np.median(r):>12.4f}{np.median(l):>12.4f}")

            raw median  log median
views           0.0009      0.4477
likes           0.0004      0.5389
comments        0.0014      0.5146


**(c) Engagement score.**

`engagement = 0.25·views_n + 0.30·likes_n + 0.25·comments_n + 0.20·like_ratio`  (score in [0, 1])

| Component | Weight | Why |
|---|---|---|
| `views_n` (reach) | 0.25 | Reach matters, but every record here has already been selected *for* views by being on the trending list, so views should not dominate. |
| `likes_n` | 0.30 | The largest weight: a like is an active, positive action, and likes scale with reach, so this rewards big **and** liked videos. |
| `comments_n` | 0.25 | Comments are the strongest form of engagement (the viewer stops to write), but they also rise for controversy, so less than likes. |
| `like_ratio` | 0.20 | A quality check: it separates videos that are liked from those that are merely seen (e.g. divisive news). NaN → 0.5 (neutral). |

The weights sum to 1, so the score stays in [0, 1] and each weight can be read as a share of the score.

In [17]:
W_VIEWS, W_LIKES, W_COMMENTS, W_RATIO = 0.25, 0.30, 0.25, 0.20
ratio_filled = np.where(np.isnan(like_ratio), 0.5, like_ratio)
engagement = W_VIEWS * views_n + W_LIKES * likes_n + W_COMMENTS * comments_n + W_RATIO * ratio_filled

print("engagement: shape", engagement.shape, "dtype", engagement.dtype)
print(f"min {engagement.min():.3f} | median {np.median(engagement):.3f} | max {engagement.max():.3f}")

engagement: shape (45904,) dtype float64
min 0.166 | median 0.586 | max 0.991


**(d) Log-scaled views.** Raw views span more than three orders of magnitude, from tens of thousands to tens of millions, and are
strongly right-skewed: on a linear axis a handful of huge videos stretch the scale and the typical video is squashed against zero.
`log10(views)` turns each ×10 into +1, so the whole range becomes readable.

In [18]:
log_views = np.log10(views)
print(f"raw views: min {views.min():,}  median {np.median(views):,.0f}  max {views.max():,}  (max/median = {views.max() / np.median(views):,.0f}x)")
print(f"log10    : min {log_views.min():.2f}  median {np.median(log_views):.2f}  max {log_views.max():.2f}")

raw views: min 1,066  median 318,600  max 360,760,164  (max/median = 1,132x)
log10    : min 3.03  median 5.50  max 8.56


**Top 10% mask.** A boolean mask marks every record whose score is at or above the 90th percentile.

In [19]:
p90 = np.percentile(engagement, 90)
top10_mask = engagement >= p90
print(f"90th percentile score = {p90:.3f}; flagged {top10_mask.sum():,} of {top10_mask.size:,} records ({top10_mask.mean():.1%})")
print(f"median views: top 10% = {np.median(views[top10_mask]):,.0f} vs rest = {np.median(views[~top10_mask]):,.0f}")

df["like_ratio"] = like_ratio
df["comment_rate"] = comment_rate
df["log_views"] = log_views
df["engagement_score"] = engagement
df["top10_engagement"] = top10_mask

90th percentile score = 0.693; flagged 4,591 of 45,904 records (10.0%)
median views: top 10% = 3,643,954 vs rest = 268,660


## 5. Reshape and broadcasting

A (days × countries) array of **total views per day**. GB and IN cover 75 days and US 80, so I keep the 75 days all three share.
The array is built once with `pivot_table`, then everything else is NumPy.

In [20]:
daily = df.pivot_table(index="trending_date", columns="country", values="views", aggfunc="sum", observed=True)[COUNTRIES]
daily = daily.dropna()                        # keep the days all three countries have
day_index = daily.index
arr = daily.to_numpy(dtype="float64")         # shape (days, countries)
print("days x countries:", arr.shape, arr.dtype, "| days:", day_index.min().date(), "->", day_index.max().date())

flat = arr.flatten()                          # 1-D copy, row by row
print("flatten:", flat.shape)
back = flat.reshape(arr.shape)                # and back again
print("reshape back equals original:", np.array_equal(back, arr))

weeks = arr.reshape(-1, 5, len(COUNTRIES))    # 75 days = 15 blocks of 5 days
print("reshaped into (blocks, days-per-block, countries):", weeks.shape)
print("total views per 5-day block (first 3 blocks, millions):")
print((weeks.sum(axis=1)[:3] / 1e6).round(1))

days x countries: (75, 3) float64 | days: 2017-11-14 -> 2018-01-27
flatten: (225,)
reshape back equals original: True
reshaped into (blocks, days-per-block, countries): (15, 5, 3)
total views per 5-day block (first 3 blocks, millions):
[[ 518.6  475.1  556.5]
 [ 782.6  887.2  645.2]
 [ 806.  1177.8  833.4]]


**Broadcasting.**
* `arr / arr.sum(axis=1, keepdims=True)` — the (days, 1) column of daily totals is stretched across the 3 countries: each country's
  **share of that day's total**.
* `arr / arr.mean(axis=0)` — the (3,) row of country averages is stretched down all days: each day **relative to that country's
  average** (1.0 = an average day).

In [21]:
share = arr / arr.sum(axis=1, keepdims=True)
relative = arr / arr.mean(axis=0)

print("shapes:", arr.shape, "/", arr.sum(axis=1, keepdims=True).shape, "->", share.shape)
print("each row of share sums to 1:", np.allclose(share.sum(axis=1), 1))
print("average share of daily views:", dict(zip(COUNTRIES, share.mean(axis=0).round(3))))
print()
print("relative to country average - first 5 days:")
print(pd.DataFrame(relative[:5], index=day_index[:5].date, columns=COUNTRIES).round(2))
print()
best = relative.argmax(axis=0)
for j, c in enumerate(COUNTRIES):
    print(f"{c}: busiest day {day_index[best[j]].date()} at {relative[best[j], j]:.2f}x its average")

shapes: (75, 3) / (75, 1) -> (75, 3)
each row of share sums to 1: True
average share of daily views: {'US': 0.294, 'GB': 0.431, 'IN': 0.275}

relative to country average - first 5 days:
              US    GB    IN
2017-11-14 0.400 0.180 0.430
2017-11-15 0.450 0.250 0.480
2017-11-16 0.510 0.310 0.570
2017-11-17 0.580 0.380 0.720
2017-11-18 0.590 0.390 0.730

US: busiest day 2017-12-18 at 1.57x its average
GB: busiest day 2018-01-27 at 2.15x its average
IN: busiest day 2017-12-11 at 1.88x its average


## 6. Joining category names

I use a **left merge** from the trending records to `categories.csv`: every trending record must survive even if its `category_id`
were missing from the lookup (it would just get a NaN name, which I check for). An inner merge would silently drop such records.
`validate="many_to_one"` also guarantees the lookup has one row per id, so the merge cannot duplicate records.

In [22]:
categories = pd.read_csv(f"{DATA_DIR}/categories.csv")
print(categories.shape)
rows_before = len(df)
df = df.merge(categories, on="category_id", how="left", validate="many_to_one")
print(f"rows before {rows_before:,} -> after {len(df):,} | records lost: {rows_before - len(df)}")
print("records without a category name:", df["category_name"].isna().sum())
unused = sorted(set(categories["category_name"]) - set(df["category_name"]))
print("categories in the lookup with no trending record:", unused)

(17, 2)
rows before 45,904 -> after 45,904 | records lost: 0
records without a category name: 0
categories in the lookup with no trending record: ['Movies', 'Nonprofits & Activism', 'Shows']


## 7. Filtering

**(a) Records with 1 million+ views *and* more than 50,000 likes** — boolean indexing with `&`, then `loc` to pick columns.

In [23]:
COLS = ["country", "trending_date", "title", "category_name", "views", "likes", "engagement_score"]
mask_a = (df["views"] >= 1_000_000) & (df["likes"] > 50_000)
big_and_liked = df.loc[mask_a, COLS].sort_values("views", ascending=False)
print(f"{len(big_and_liked):,} records ({mask_a.mean():.1%} of all)")
big_and_liked.head(10)

6,707 records (14.6% of all)


,country,trending_date,title,category_name,views,likes,engagement_score
6698,GB,2018-01-05,Lena Ortiz - Monsoon (Live),Music,360760164,27537822,0.991
6473,GB,2018-01-27,The Hollow Pines - Phones (Official Video) WATCH,Music,343297002,9581910,0.972
6697,GB,2018-01-04,Lena Ortiz - Monsoon (Live),Music,286714357,21885701,0.978
6472,GB,2018-01-26,The Hollow Pines - Phones (Official Video) WATCH,Music,282812154,7893691,0.962
7920,GB,2017-12-24,Rook & Vale - Phones (Official Video) WATCH,Music,255362869,11413793,0.968
6696,GB,2018-01-03,Lena Ortiz - Monsoon (Live),Music,236275528,18035565,0.968
6471,GB,2018-01-25,The Hollow Pines - Phones (Official Video) WATCH,Music,230704137,6439282,0.950
6695,GB,2018-01-02,Lena Ortiz - Monsoon (Live),Music,210747294,16086924,0.961
7919,GB,2017-12-23,Rook & Vale - Phones (Official Video) WATCH,Music,200444300,8959133,0.954
6470,GB,2018-01-24,The Hollow Pines - Phones (Official Video) WATCH,Music,181785943,5073905,0.937


**(b) Top 20 records by engagement score** — sort, then `iloc` for the first 20 positions.

In [24]:
df.sort_values("engagement_score", ascending=False).iloc[:20][COLS]

,country,trending_date,title,category_name,views,likes,engagement_score
6698,GB,2018-01-05,Lena Ortiz - Monsoon (Live),Music,360760164,27537822,0.991
6697,GB,2018-01-04,Lena Ortiz - Monsoon (Live),Music,286714357,21885701,0.978
6473,GB,2018-01-27,The Hollow Pines - Phones (Official Video) WATCH,Music,343297002,9581910,0.972
6696,GB,2018-01-03,Lena Ortiz - Monsoon (Live),Music,236275528,18035565,0.968
7920,GB,2017-12-24,Rook & Vale - Phones (Official Video) WATCH,Music,255362869,11413793,0.968
6472,GB,2018-01-26,The Hollow Pines - Phones (Official Video) WATCH,Music,282812154,7893691,0.962
6695,GB,2018-01-02,Lena Ortiz - Monsoon (Live),Music,210747294,16086924,0.961
7919,GB,2017-12-23,Rook & Vale - Phones (Official Video) WATCH,Music,200444300,8959133,0.954
6471,GB,2018-01-25,The Hollow Pines - Phones (Official Video) WATCH,Music,230704137,6439282,0.950
6694,GB,2018-01-01,Lena Ortiz - Monsoon (Live),Music,169181540,12914095,0.949


**(c) Music *or* Gaming records with views above the overall median** — `|` inside the category condition, `&` with the views condition.

In [25]:
median_views = df["views"].median()
mask_c = ((df["category_name"] == "Music") | (df["category_name"] == "Gaming")) & (df["views"] > median_views)
music_gaming = df.loc[mask_c, COLS]
print(f"overall median views = {median_views:,.0f}; matching records: {mask_c.sum():,}")
print(music_gaming["category_name"].value_counts())
music_gaming.sort_values("views", ascending=False).head(10)

overall median views = 318,600; matching records: 9,106
category_name
Music     8243
Gaming     863
Name: count, dtype: int64


,country,trending_date,title,category_name,views,likes,engagement_score
6698,GB,2018-01-05,Lena Ortiz - Monsoon (Live),Music,360760164,27537822,0.991
6473,GB,2018-01-27,The Hollow Pines - Phones (Official Video) WATCH,Music,343297002,9581910,0.972
6697,GB,2018-01-04,Lena Ortiz - Monsoon (Live),Music,286714357,21885701,0.978
6472,GB,2018-01-26,The Hollow Pines - Phones (Official Video) WATCH,Music,282812154,7893691,0.962
7920,GB,2017-12-24,Rook & Vale - Phones (Official Video) WATCH,Music,255362869,11413793,0.968
6696,GB,2018-01-03,Lena Ortiz - Monsoon (Live),Music,236275528,18035565,0.968
6471,GB,2018-01-25,The Hollow Pines - Phones (Official Video) WATCH,Music,230704137,6439282,0.950
6695,GB,2018-01-02,Lena Ortiz - Monsoon (Live),Music,210747294,16086924,0.961
7919,GB,2017-12-23,Rook & Vale - Phones (Official Video) WATCH,Music,200444300,8959133,0.954
6470,GB,2018-01-24,The Hollow Pines - Phones (Official Video) WATCH,Music,181785943,5073905,0.937


## 8. Aggregation

`groupby` + `agg` with named aggregations, by category and by country.

In [26]:
AGG = dict(records=("video_id", "size"), avg_views=("views", "mean"),
           avg_likes=("likes", "mean"), avg_engagement=("engagement_score", "mean"))
by_category = df.groupby("category_name").agg(**AGG).sort_values("avg_views", ascending=False)
by_category

,records,avg_views,avg_likes,avg_engagement
category_name,,,,
Music,10286,"3,193,066.328","135,046.350",0.644
Entertainment,12242,"902,097.344","28,980.808",0.591
Comedy,3251,"602,812.395","28,658.117",0.593
Film & Animation,2290,"578,834.931","17,218.499",0.564
People & Blogs,3797,"551,729.444","19,849.883",0.576
Gaming,2055,"509,738.929","20,778.595",0.585
Sports,1919,"374,671.119","10,452.668",0.551
Education,1643,"362,894.163","12,626.386",0.544
Science & Technology,1558,"356,498.153","11,358.379",0.550


In [27]:
by_country = df.groupby("country", observed=True).agg(**AGG)
by_country

,records,avg_views,avg_likes,avg_engagement
country,,,,
GB,14969,"1,584,054.416","65,276.013",0.587
IN,14967,"953,371.238","33,196.635",0.589
US,15968,"1,014,564.291","39,561.197",0.582


In [28]:
by_cat_country = df.groupby(["category_name", "country"], observed=True).agg(**AGG).round(3)
by_cat_country.loc[["Music", "Entertainment", "News & Politics"]]

records     avg_views   avg_likes  avg_engagement
category_name   country                                                   
Music           GB          4978 3,729,081.105 161,427.909           0.642
                IN          2279 2,451,013.283  93,420.936           0.648
                US          3029 2,870,470.529 123,008.355           0.646
Entertainment   GB          2820   818,276.041  26,832.768           0.580
                IN          5080   966,402.356  31,259.241           0.598
                US          4342   881,301.998  27,710.204           0.590
News & Politics GB           533   233,975.919   2,472.060           0.485
                IN          1134   346,449.767   3,882.983           0.508
                US           751   225,067.631   2,411.788           0.487

### Category distribution — `value_counts()` as counts and percentages

In [29]:
dist = pd.DataFrame({"records": df["category_name"].value_counts(),
                     "percent": (df["category_name"].value_counts(normalize=True) * 100).round(2)})
dist

,records,percent
category_name,,
Entertainment,12242,26.670
Music,10286,22.410
People & Blogs,3797,8.270
Comedy,3251,7.080
News & Politics,2418,5.270
Film & Animation,2290,4.990
Gaming,2055,4.480
Sports,1919,4.180
Howto & Style,1819,3.960


### When are the highest-viewed videos published?
Each video appears on many days with growing views, so I compare **videos**, not records: one row per (country, video) with its peak
views, then group by the publish weekday and hour. The median is reported next to the mean because a few huge videos move the mean.

In [30]:
videos = (df.groupby(["country", "video_id"], observed=True)
            .agg(publish_time=("publish_time", "first"), peak_views=("views", "max"))
            .reset_index())
videos["publish_day"] = videos["publish_time"].dt.day_name()
videos["publish_hour"] = videos["publish_time"].dt.hour

by_day = (videos.groupby("publish_day")["peak_views"].agg(["count", "mean", "median"])
          .sort_values("mean", ascending=False))
by_day

,count,mean,median
publish_day,,,
Monday,2151,"1,192,554.451","272,614.000"
Friday,2073,"1,021,225.582","258,678.000"
Tuesday,2290,"1,009,725.123","267,736.500"
Wednesday,2165,"933,784.894","272,569.000"
Thursday,2122,"929,178.118","242,952.500"
Saturday,2153,"850,825.871","266,608.000"
Sunday,2089,"786,431.495","256,768.000"


In [31]:
by_hour = videos.groupby("publish_hour")["peak_views"].agg(["count", "mean", "median"])
print("Top 5 publish hours (UTC) by mean peak views:")
print(by_hour.sort_values("mean", ascending=False).head(5).round(0))
print(f"\nspread of hourly means: {by_hour['mean'].min():,.0f} - {by_hour['mean'].max():,.0f}")

Top 5 publish hours (UTC) by mean peak views:
              count          mean      median
publish_hour                                 
12              606 2,106,329.000 264,636.000
8               624 1,392,749.000 257,374.000
16              637 1,197,658.000 282,269.000
14              618 1,106,425.000 239,347.000
21              651 1,089,490.000 273,063.000

spread of hourly means: 671,835 - 2,106,329


**Result.** Videos published on **Monday** have the highest mean peak views (≈1.19 M vs ≈0.79 M for Sunday), and **12:00 UTC** is
the best publishing hour by mean (≈2.1 M). But the *medians* are almost identical across weekdays (≈243 k–273 k) and hours, so these
mean differences come from a few very large videos that happened to be published then. Publish timing is a weak lever in this data
compared with category.

## 9. Time to trend and trending lifespan

`days_to_trend` = calendar days from `publish_time` to `trending_date` (both normalised to the date).

In [32]:
df["days_to_trend"] = (df["trending_date"] - df["publish_time"].dt.normalize()).dt.days
first_trend = df.sort_values("trending_date").drop_duplicates(VIDEO)       # each video's first trending day
print("days_to_trend on the FIRST trending day of each (country, video):")
print(first_trend["days_to_trend"].describe().round(2))
print()
print("share of videos trending within N days of publishing:")
for n in [0, 1, 2, 3, 7]:
    print(f"  <= {n} days: {(first_trend['days_to_trend'] <= n).mean():.1%}")

days_to_trend on the FIRST trending day of each (country, video):


count   15,043.000
mean         2.800
std          5.630
min          0.000
25%          0.000
50%          1.000
75%          2.000
max         38.000
Name: days_to_trend, dtype: float64

share of videos trending within N days of publishing:
  <= 0 days: 40.7%
  <= 1 days: 63.3%
  <= 2 days: 75.8%
  <= 3 days: 82.4%
  <= 7 days: 90.6%


**Trending lifespan** = number of distinct days a video spent on one country's list (`groupby` + `nunique`).

In [33]:
lifespan = (df.groupby(["country", "video_id"], observed=True)
              .agg(lifespan_days=("trending_date", "nunique"), title=("title", "first"),
                   category_name=("category_name", "first"), channel_title=("channel_title", "first"),
                   peak_views=("views", "max"))
              .reset_index())
print(lifespan["lifespan_days"].describe().round(2))
print("\n10 longest-lasting videos:")
lifespan.sort_values(["lifespan_days", "peak_views"], ascending=False).head(10)

count   15,043.000
mean         3.050
std          2.920
min          1.000
25%          1.000
50%          2.000
75%          4.000
max         25.000
Name: lifespan_days, dtype: float64

10 longest-lasting videos:


,country,video_id,lifespan_days,title,category_name,channel_title,peak_views
1845,GB,RdqSDDng84n,25,Lena Ortiz - Monsoon (Live),Music,FinalSpeech81,360760164
11892,US,Nc1MUQvmyWO,25,Sunny Park - Cats (Official Video),Music,ShockingInterview30,97553795
1611,GB,NqokCmZykzK,25,Anya Lux - Paris (Lyric Video),Music,QuietInterview9,97513776
2686,GB,dIbJkzpClRM,25,Sunny Park - Robots (Live),Music,SurprisingUpdate52,60754408
2667,GB,cz0dTlZWwSk,25,Final Cars Speech WATCH,Film & Animation,UltimateUnboxing65,60717690
10992,US,BsTMdCTYs7P,25,Surprising Guitar Update,Entertainment,Brand NewUnboxing21,54982508
1232,GB,Hh9b0BBsne0,25,Kairo - Phones (Lyric Video)!,Music,PerfectTrailer65,43802674
1019,GB,EZaG9_2Yi1u,25,KAIRO - FESTIVAL (LIVE),Music,QuietChallenge75,38242672
950,GB,DYLzrqG-SrK,25,Official Football Update,Entertainment,InsaneReaction46,26603358
1630,GB,O6zdCRwWdf1,25,MIRA SOLIS - WINTER (LYRIC VIDEO)?,Music,FinalTrailer17,17542788


In [34]:
life_by_cat = (lifespan.groupby("category_name")["lifespan_days"]
               .agg(videos="size", avg_lifespan="mean", median_lifespan="median")
               .sort_values("avg_lifespan", ascending=False).round(2))
life_by_cat

,videos,avg_lifespan,median_lifespan
category_name,,,
Music,1932,5.320,4.000
Entertainment,3463,3.540,3.000
Comedy,1166,2.790,2.000
People & Blogs,1364,2.780,2.000
Film & Animation,847,2.700,2.000
Gaming,778,2.640,2.000
Education,673,2.440,2.000
Howto & Style,758,2.400,2.000
Autos & Vehicles,425,2.370,2.000


## 10. Combining data

**`concat`** — the combined frame in section 1 was built with `pd.concat()`; here I confirm the stack is complete and the per-country
pieces can be recovered from it.

In [35]:
check = pd.concat(frames.values(), keys=frames.keys(), names=["source", "row"])
print("concat with keys:", check.shape, "| rows per source:", check.groupby(level="source").size().to_dict())
print("matches the combined raw frame:", len(check) == len(raw))

concat with keys: (46230, 17) | rows per source: {'GB': 15075, 'IN': 15075, 'US': 16080}
matches the combined raw frame: True


**`merge`** — a per-video summary table and a per-channel summary table, joined on `channel_title`.
* `video_summary`: one row per unique `video_id` with its channel, peak views (max over all countries/days) and number of countries.
* `channel_summary`: built from the trending records — number of trending records, number of countries and the channel's main category.

Grouping `video_summary` by channel gives *number of trending videos* and *total views*; merging that with `channel_summary` puts all the
channel facts in one table. It is a one-to-one merge on channel, validated.

In [36]:
video_summary = (df.groupby("video_id")
                   .agg(channel_title=("channel_title", "first"), peak_views=("views", "max"),
                        countries=("country", "nunique"))
                   .reset_index())

channel_summary = (df.groupby("channel_title")
                     .agg(trending_records=("video_id", "size"), countries=("country", "nunique"),
                          main_category=("category_name", lambda s: s.mode().iat[0]))
                     .reset_index())

per_channel_videos = (video_summary.groupby("channel_title")
                        .agg(trending_videos=("video_id", "nunique"), total_views=("peak_views", "sum"))
                        .reset_index())

channels = per_channel_videos.merge(channel_summary, on="channel_title", how="inner", validate="one_to_one")
print("video_summary:", video_summary.shape, "| channel_summary:", channel_summary.shape, "| merged:", channels.shape)
channels.sort_values("total_views", ascending=False).head(15)

video_summary: (14684, 4) | channel_summary: (438, 4) | merged: (438, 6)


,channel_title,trending_videos,total_views,trending_records,countries,main_category
290,QuietChallenge75,466,1333310585,2448,3,Music
432,WildUnboxing39,56,582453017,394,3,Music
133,FinalTrailer17,161,492211013,902,3,Music
281,PerfectTutorial78,98,477526034,537,3,Music
278,PerfectTrailer65,66,453035336,339,3,Music
132,FinalSpeech81,34,442854302,185,3,Music
359,SurprisingHighlights41,123,401045203,648,3,Music
248,OfficialReview41,457,392673620,1575,3,Entertainment
294,QuietInterview9,88,342109758,564,3,Music
31,Brand NewChallenge45,302,305611036,1169,3,Entertainment


**Takeaways from this Part**
* After cleaning, the table holds one row per country × video × day, with no missing values in the numeric columns.
* Entertainment (26.7% of records) and Music (22.4%) dominate the lists; Music also has by far the highest average views (≈3.19 M per record).
* Music videos stay longest (average lifespan 5.3 days) while News & Politics is shortest (1.7 days) and has the lowest engagement score (0.50).
* 63.3% of videos reach the trending list within 1 day of publishing and 90.6% within a week (median 1 day).
* The engagement score's top 10% (score ≥ 0.693) have a median of ≈3.6 M views versus ≈0.27 M for the rest.